In [17]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [18]:
stop_words = set(stopwords.words("english"))
print(stop_words)

{'here', 'than', 'are', 'those', 'once', 'hers', "he'll", 'so', 'but', 'myself', 'these', "you'll", 'if', 'yourselves', 'own', 'was', 'will', 'doesn', 'more', 'has', 'out', 'haven', 'during', 'ours', 'having', 'as', "it's", 'any', 'him', 'have', 'down', 'the', 'd', 'm', "he'd", 'her', "should've", "couldn't", 'y', 'all', "isn't", 'no', 'our', 'most', 'wouldn', 'am', 'between', "she'll", 'them', 'other', "aren't", "mightn't", 'should', 'their', "they'd", 'where', "they'll", 'yourself', 'this', "i'll", "we're", "won't", 'under', 'or', 've', "i've", "they're", 'that', 'into', 'my', "she's", 'such', "you'd", 'his', 'above', "didn't", 'both', "we'd", 'mightn', 'don', "don't", 'not', "you're", 'they', 'and', "it'll", "hasn't", 'we', 'doing', 'from', "he's", 'herself', 'an', 'ain', 'it', 'while', 'who', 'i', "i'm", 'll', 'won', 'she', 'shouldn', 'himself', 'now', "hadn't", 'how', "i'd", 't', "that'll", 'below', 'do', "haven't", 'weren', 'at', 'had', 'couldn', "we've", "wasn't", 'being', 'hasn

In [19]:
df = pd.read_csv("../dataset/Suicide_Detection.csv")
df.head()

,Unnamed: 0,text,class
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,3,Am I weird I don't get affected by compliments...,non-suicide
2,4,Finally 2020 is almost over... So I can never ...,non-suicide
3,8,i need helpjust help me im crying so hard,suicide
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [20]:
df = df.drop(columns=["Unnamed: 0"])
df.head()

,text,class
0,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,Am I weird I don't get affected by compliments...,non-suicide
2,Finally 2020 is almost over... So I can never ...,non-suicide
3,i need helpjust help me im crying so hard,suicide
4,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [21]:
print(df.shape)
print("-"*100)
print(df.info())
print("-"*100)
print(df.describe())
print("-"*100)
print(df.isnull().sum())
print("-"*100)

(232074, 2)
----------------------------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232074 entries, 0 to 232073
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    232074 non-null  object
 1   class   232074 non-null  object
dtypes: object(2)
memory usage: 3.5+ MB
None
----------------------------------------------------------------------------------------------------
                                                     text    class
count                                              232074   232074
unique                                             232074        2
top     I still haven't beaten the first boss in Hollo...  suicide
freq                                                    1   116037
----------------------------------------------------------------------------------------------------
text     0
class    0
dtype: int64
------------

In [22]:
df.duplicated().sum()

np.int64(0)

In [23]:
df["class"].value_counts()

class
suicide        116037
non-suicide    116037
Name: count, dtype: int64

In [24]:
# stop_words = set(stopwords.words("english"))

# keep_words = {
#     "not", "no", "nor", "never",
#     "don't", "didn't", "doesn't", "won't", "can't",
#     "couldn't", "shouldn't", "wouldn't",

#     "i", "me", "my", "myself",
#     "we", "us", "our", "ourselves",

#     "very", "really", "so", "too", "extremely", "highly", "deeply",

#     "want", "need", "wish", "hope", "trying", "going", "planning",
#     "should", "must", "will"
# }
# stop_words = stop_words - keep_words

# lemmatizer = WordNetLemmatizer()

# def clean_words(text):
#     text = str(text).lower()
#     text = re.sub(r"http\S+|www\S+", "", text)
#     text = re.sub(r"[^a-zA-Z']", " ", text)
#     tokens = word_tokenize(text) 
#     cleaned_tokens = []
#     for word in tokens:
#         if word not in stop_words:
#             word = lemmatizer.lemmatize(word)
#             cleaned_tokens.append(word)
    
#     return " ".join(cleaned_tokens)

def clean_words(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [25]:
df["clean_text"] = df["text"].apply(clean_words)

In [26]:
df.head()

,text,class,clean_text
0,Ex Wife Threatening SuicideRecently I left my ...,suicide,ex wife threatening suiciderecently i left my ...
1,Am I weird I don't get affected by compliments...,non-suicide,am i weird i don t get affected by compliments...
2,Finally 2020 is almost over... So I can never ...,non-suicide,finally is almost over so i can never hear has...
3,i need helpjust help me im crying so hard,suicide,i need helpjust help me im crying so hard
4,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,i m so losthello my name is adam and i ve been...


In [27]:
df = df[df["clean_text"].str.strip() != ""]

In [28]:
df.shape

(232010, 3)

In [29]:
df["labels"] = df["class"].map({"suicide": 1, "non-suicide": 0})

In [30]:
df.head()

,text,class,clean_text,labels
0,Ex Wife Threatening SuicideRecently I left my ...,suicide,ex wife threatening suiciderecently i left my ...,1
1,Am I weird I don't get affected by compliments...,non-suicide,am i weird i don t get affected by compliments...,0
2,Finally 2020 is almost over... So I can never ...,non-suicide,finally is almost over so i can never hear has...,0
3,i need helpjust help me im crying so hard,suicide,i need helpjust help me im crying so hard,1
4,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,i m so losthello my name is adam and i ve been...,1


In [31]:
df["labels"].value_counts()

labels
1    116027
0    115983
Name: count, dtype: int64

In [32]:
df.to_csv("../dataset/Cleaned_Suicide_Detection_DL.csv", index=False)